# 08 - Persistencia en Firebase / Firestore

Este notebook documenta y prepara la carga de metadata del dataset, metricas de modelos, resultados comparativos y configuraciones experimentales en Firestore. No reentrena modelos y no modifica archivos en `data/raw/`.

## Por que Firestore

Firestore es una base NoSQL orientada a documentos. Es adecuada para esta etapa porque los resultados del proyecto tienen estructura semiestructurada: metadata del dataset, metricas por experimento, configuraciones de entrenamiento y resultados de validacion cruzada. Estos documentos pueden evolucionar sin exigir un esquema relacional rigido.

Persistir metricas y configuracion es clave para la trazabilidad del experimento. No alcanza con saber cual fue el mejor modelo: tambien se necesita guardar con que dataset, features, target, particion, pesos de clase y criterio de seleccion se obtuvo ese resultado.

## Colecciones creadas

- `datasets`: metadata del dataset procesado `siniestros_limpio_enriquecido`.
- `model_results`: un documento por experimento/modelo con metricas principales.
- `model_config`: un documento por configuracion experimental con features, preprocessing, split y notas metodologicas.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.upload_results_firebase import (
    DATA_FILE,
    LOG_FILE,
    build_firestore_payloads,
    upload_payloads,
)
from src.firebase_client import get_pipeline_logger, load_env_file

pd.set_option("display.max_columns", 100)

## Revision de metadata local

Antes de subir a Firestore se lee el CSV procesado para obtener `shape`, columnas y tipos de datos. Esta metadata permite saber exactamente que version analitica del dataset acompana a los resultados de modelado.

In [2]:
df_metadata = pd.read_csv(DATA_FILE, nrows=5)
full_shape = pd.read_csv(DATA_FILE).shape

print(f"Dataset procesado: {DATA_FILE.resolve()}")
print(f"Shape: {full_shape}")
display(df_metadata.dtypes.rename("dtype").to_frame())

Dataset procesado: C:\Users\Germán\Desktop\TP AVANZADA\tp-final-siniestros-viales\data\processed\siniestros_limpio_enriquecido.csv
Shape: (62076, 14)


,dtype
fecha_siniestro,object
anio_siniestro,int64
modo_desplazamiento_victima,object
sexo_victima,object
edad_victima,int64
gravedad_victima,object
rol_victima,float64
edad_grupo,object
es_mortal,int64
es_grave_o_mortal,int64


## Construccion de documentos

Los documentos se construyen a partir de los JSON ya generados en `outputs/`: `model_metrics.json`, `model_comparison.json` y `cross_validation_results.json`. Esta etapa solo persiste resultados existentes; no reentrena modelos.

In [3]:
payloads = build_firestore_payloads()

print("Documento datasets:")
display(pd.DataFrame([payloads["dataset"]]))

print("Documentos model_results:")
display(pd.DataFrame(payloads["model_results"]))

print("Documentos model_config:")
display(pd.DataFrame(payloads["model_config"]))

Documento datasets:


,nombre,fecha_carga,cantidad_filas,cantidad_columnas,columnas,dtypes,fuente,version_dataset
0,siniestros_limpio_enriquecido,2026-05-30T23:33:53.482895+00:00,62076,14,"[fecha_siniestro, anio_siniestro, modo_desplaz...","{'fecha_siniestro': 'object', 'anio_siniestro'...",data\processed\siniestros_limpio_enriquecido.csv,v1_enriquecido


Documentos model_results:


,experiment_id,model_name,target,accuracy,precision,recall,f1,f1_mean,f1_std,cv_folds,selected_model,created_at,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std
0,baseline_logistic_regression,LogisticRegression,es_grave_o_mortal,0.955380,1.000000,0.100649,0.182891,NaN,NaN,NaN,False,2026-05-30T23:33:53.482895+00:00,NaN,NaN,NaN,NaN,NaN,NaN
1,comparison_RandomForestClassifier,RandomForestClassifier,es_grave_o_mortal,0.720361,0.139576,0.897727,0.241590,NaN,NaN,NaN,True,2026-05-30T23:33:53.482895+00:00,NaN,NaN,NaN,NaN,NaN,NaN
2,comparison_DecisionTreeClassifier,DecisionTreeClassifier,es_grave_o_mortal,0.715287,0.136850,0.892857,0.237325,NaN,NaN,NaN,False,2026-05-30T23:33:53.482895+00:00,NaN,NaN,NaN,NaN,NaN,NaN
3,comparison_LogisticRegression,LogisticRegression,es_grave_o_mortal,0.651015,0.118299,0.935065,0.210027,NaN,NaN,NaN,False,2026-05-30T23:33:53.482895+00:00,NaN,NaN,NaN,NaN,NaN,NaN
4,cross_validation_RandomForestClassifier,RandomForestClassifier,es_grave_o_mortal,NaN,NaN,NaN,NaN,0.235999,0.001846,5.0,True,2026-05-30T23:33:53.482895+00:00,0.719215,0.005921,0.136434,0.001191,0.873731,0.018261
5,cross_validation_DecisionTreeClassifier,DecisionTreeClassifier,es_grave_o_mortal,NaN,NaN,NaN,NaN,0.226749,0.007606,5.0,False,2026-05-30T23:33:53.482895+00:00,0.700625,0.015789,0.130123,0.005168,0.882824,0.015452
6,cross_validation_LogisticRegression,LogisticRegression,es_grave_o_mortal,NaN,NaN,NaN,NaN,0.206088,0.001239,5.0,False,2026-05-30T23:33:53.482895+00:00,0.647610,0.006089,0.116022,0.000856,0.921447,0.011989


Documentos model_config:


,experiment_id,target,features,excluded_features,preprocessing,train_test_split,random_state,class_weight,notes
0,baseline_logistic_regression,es_grave_o_mortal,"{'numeric': ['anio_siniestro', 'mes_siniestro'...","[GRAVEdad_victima, gravedad_victima, es_mortal...",{'categorical': 'OneHotEncoder(handle_unknown=...,"{'test_size': 0.2, 'stratify': True}",42,None,Primer baseline LogisticRegression sin class_w...
1,model_comparison,es_grave_o_mortal,"{'numeric': ['anio_siniestro', 'mes_siniestro'...","[GRAVEdad_victima, gravedad_victima, es_mortal...",{'categorical': 'OneHotEncoder(handle_unknown=...,"{'test_size': 0.2, 'stratify': True}",42,balanced,"Comparacion LogisticRegression, DecisionTreeCl..."
2,cross_validation,es_grave_o_mortal,"{'numeric': ['anio_siniestro', 'mes_siniestro'...","[GRAVEdad_victima, gravedad_victima, es_mortal...",{'categorical': 'OneHotEncoder(handle_unknown=...,None,42,balanced,Validacion con StratifiedKFold de 5 folds; sel...


## Carga a Firestore

La carga real requiere credenciales locales. No deben subirse al repo. Configure un archivo `.env` ignorado por Git con `FIREBASE_CREDENTIALS_PATH` o use `GOOGLE_APPLICATION_CREDENTIALS` como variable de entorno.

Por seguridad, `RUN_UPLOAD` queda en `False`. Para ejecutar la escritura desde notebook, cambiarlo a `True` en un entorno local con credenciales configuradas.

In [4]:
RUN_UPLOAD = False

logger = get_pipeline_logger(LOG_FILE)
logger.info("Inicio de carga a Firebase/Firestore desde notebook")
load_env_file(PROJECT_ROOT / ".env")

if RUN_UPLOAD:
    upload_payloads(payloads, logger)
    logger.info("Carga a Firebase/Firestore desde notebook finalizada correctamente")
else:
    logger.info("Notebook ejecutado en modo revision; no se escribio en Firestore")
    print("Modo revision: no se escribio en Firestore.")

INFO | Inicio de carga a Firebase/Firestore desde notebook


INFO | Notebook ejecutado en modo revision; no se escribio en Firestore


Modo revision: no se escribio en Firestore.


## Ejecucion recomendada por script

Para produccion local se recomienda ejecutar el script versionado:

```bash
python scripts/upload_results_firebase.py --dry-run
python scripts/upload_results_firebase.py --execute
```

El primer comando valida archivos y payloads sin escribir. El segundo crea o actualiza los documentos en Firestore.